In [62]:
import pandas as pd
import datetime
from pandasql import sqldf
import sqlite3
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier  # Or another classifier
from sklearn.metrics import classification_report
from sklearn.preprocessing import OneHotEncoder  # Use OneHotEncoder




pd.options.display.max_rows = 50
pd.options.display.max_columns = 100

mysqldf = lambda q: sqldf(q, globals())

conn = sqlite3.connect('optionsQuotes1.db')
#c = conn.cursor()
df = pd.read_sql('select * from data',conn)
conn.close()
df['time_converted'] = pd.to_datetime(df['time_converted'])

In [3]:
df.tail(10)

,index,volume_options,volume_weighted_options,open_options,close_options,high_options,low_options,timestamp_options,number of trades_options,ticker,full_name,type,strike,expiry,equity_start_price,time_converted,open_equity,close_equity,high_equity,low_equity,timestamp_equity,number of trades_equity,_merge,equity_pct_change,options_pct_change,options_earliest_open,file_source,equity_pct_change_normalized,day_classification,DTE,day_name,DTE_adjusted
718705,718705,11,0.0109,0.01,0.02,0.02,0.01,1701287100000,2,SPY,O:SPY231201C00470000,call,470,2023-12-01 00:00:00,457.6,2023-11-29 14:45:00,455.230,455.5250,455.560,455.190,1.701287e+12,10229.0,both,0.994821,0.500000,0.02,0-2DTE_spy_options_01Sep23-31Dec23.pkl,-0.517920,average,2,Wednesday,2
718706,718706,12,0.0100,0.01,0.01,0.01,0.01,1701288900000,1,SPY,O:SPY231201C00470000,call,470,2023-12-01 00:00:00,457.6,2023-11-29 15:15:00,455.120,454.4400,455.180,454.370,1.701289e+12,21456.0,both,0.994580,0.500000,0.02,0-2DTE_spy_options_01Sep23-31Dec23.pkl,-0.541958,average,2,Wednesday,2
718707,718707,2,0.0100,0.01,0.01,0.01,0.01,1701289800000,1,SPY,O:SPY231201C00470000,call,470,2023-12-01 00:00:00,457.6,2023-11-29 15:30:00,454.440,454.5801,454.630,454.200,1.701290e+12,26984.0,both,0.993094,0.500000,0.02,0-2DTE_spy_options_01Sep23-31Dec23.pkl,-0.690559,average,2,Wednesday,2
718708,718708,1,14.0400,14.04,14.04,14.04,14.04,1701277200000,1,SPY,O:SPY231201P00470000,put,470,2023-12-01 00:00:00,457.6,2023-11-29 12:00:00,455.765,455.9400,456.030,455.550,1.701277e+12,8899.0,both,0.995990,1.000000,14.04,0-2DTE_spy_options_01Sep23-31Dec23.pkl,-0.401005,average,2,Wednesday,2
718709,718709,45,14.1831,14.18,14.19,14.19,14.18,1701286200000,2,SPY,O:SPY231201P00470000,put,470,2023-12-01 00:00:00,457.6,2023-11-29 14:30:00,455.890,455.2205,455.890,455.090,1.701286e+12,13493.0,both,0.996263,1.009972,14.04,0-2DTE_spy_options_01Sep23-31Dec23.pkl,-0.373689,average,2,Wednesday,2
718710,718710,60,14.8070,14.78,14.81,14.81,14.78,1701287100000,2,SPY,O:SPY231201P00470000,put,470,2023-12-01 00:00:00,457.6,2023-11-29 14:45:00,455.230,455.5250,455.560,455.190,1.701287e+12,10229.0,both,0.994821,1.052707,14.04,0-2DTE_spy_options_01Sep23-31Dec23.pkl,-0.517920,average,2,Wednesday,2
718711,718711,3,0.0167,0.02,0.01,0.02,0.01,1701271800000,2,SPY,O:SPY231201C00471000,call,471,2023-12-01 00:00:00,457.6,2023-11-29 10:30:00,456.480,456.4100,456.650,456.110,1.701272e+12,17957.0,both,0.997552,1.000000,0.02,0-2DTE_spy_options_01Sep23-31Dec23.pkl,-0.244755,average,2,Wednesday,2
718712,718712,10,0.0100,0.01,0.01,0.01,0.01,1701282600000,1,SPY,O:SPY231201C00471000,call,471,2023-12-01 00:00:00,457.6,2023-11-29 13:30:00,456.295,455.9999,456.310,455.695,1.701283e+12,9800.0,both,0.997148,0.500000,0.02,0-2DTE_spy_options_01Sep23-31Dec23.pkl,-0.285184,average,2,Wednesday,2
718713,718713,10,0.0100,0.01,0.01,0.01,0.01,1701285300000,1,SPY,O:SPY231201C00471000,call,471,2023-12-01 00:00:00,457.6,2023-11-29 14:15:00,456.170,455.8850,456.265,455.850,1.701285e+12,9116.0,both,0.996875,0.500000,0.02,0-2DTE_spy_options_01Sep23-31Dec23.pkl,-0.312500,average,2,Wednesday,2
718714,718714,100,0.0100,0.01,0.01,0.01,0.01,1701290700000,1,SPY,O:SPY231201C00471000,call,471,2023-12-01 00:00:00,457.6,2023-11-29 15:45:00,454.590,454.6200,454.670,454.380,1.701291e+12,43099.0,both,0.993422,0.500000,0.02,0-2DTE_spy_options_01Sep23-31Dec23.pkl,-0.657780,average,2,Wednesday,2


In [23]:
daily_summary  = mysqldf(""" 
             SELECT
DATE(Time_Converted) AS 'Date',
MAX(CASE WHEN TYPE = 'call' THEN options_pct_change ELSE NULL END) AS 'Call_Max_Daily',
MAX(CASE WHEN TYPE = 'put' THEN options_pct_change ELSE NULL END) AS 'Put_Max_Daily'
FROM df
WHERE DTE_ADJUSTED = 0 AND options_earliest_open>0.2
GROUP BY DATE(Time_Converted)
        """)
daily_summary['Max'] = daily_summary[['Call_Max_Daily', 'Put_Max_Daily']].max(axis=1)
daily_summary['Date'] = pd.to_datetime(daily_summary['Date'])
daily_summary['weekday'] = daily_summary['Date'].dt.day_name()

In [29]:
# day classification for ranges. below range boundaries are in absolute pct. Ex: 1 - 1%
very_high_range = 4
high_range = 2
average_range = 1.5


def day_classification(row):
    if row['Max'] > very_high_range:
        return 'very_high'
    elif row['Max'] > high_range:
        return 'high'
    elif row['Max'] > average_range:
        return 'average'
    else:
        return 'low'


daily_summary['day_classification'] = None
daily_summary['day_classification'] = daily_summary.apply(day_classification, axis=1)

In [34]:
daily_summary_filtered = daily_summary[daily_summary['Date'] > '2023-01-12']
fig = px.histogram(daily_summary_filtered, x='day_classification', title='Distribution of Day Classification', histnorm='percent')
fig.show()

In [35]:
daily_summary_filtered['day_classification'].value_counts(normalize=True) * 100

day_classification
high         37.037037
very_high    30.555556
average      23.148148
low           9.259259
Name: proportion, dtype: float64

In [111]:
df

,Date,Call_Max_Daily,Put_Max_Daily,Max,weekday,day_classification,d1,d2,d3,weekday_encoded,week_number_in_month
42,2023-03-06,2.461538,1.559140,2.461538,Monday,high,very_high,very_high,average,1,0
43,2023-03-07,1.194030,5.571429,5.571429,Tuesday,very_high,high,very_high,very_high,3,0
44,2023-03-08,1.484848,1.303150,1.484848,Wednesday,low,very_high,high,very_high,4,0
45,2023-03-09,1.391304,27.095238,27.095238,Thursday,very_high,low,very_high,high,2,0
46,2023-03-10,2.131579,1.951456,2.131579,Friday,high,very_high,low,very_high,0,0
...,...,...,...,...,...,...,...,...,...,...,...
219,2023-11-22,1.069966,2.871795,2.871795,Wednesday,high,average,very_high,average,4,0
220,2023-11-24,1.346939,1.042038,1.346939,Friday,low,high,average,very_high,0,0
221,2023-11-27,1.612903,1.019108,1.612903,Monday,average,low,high,average,1,0
222,2023-11-28,4.509804,1.215000,4.509804,Tuesday,very_high,average,low,high,3,0


In [192]:
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import GridSearchCV

import numpy as np

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}


def create_lags(df):
    df['d1'] = df.loc[:,'day_classification'].shift(1)
    df['d2'] = df.loc[:,'day_classification'].shift(2)
    df['d3'] = df.loc[:,'day_classification'].shift(3)
    df['weekday_encoded'] = LabelEncoder().fit_transform(df['weekday'])  # Encode weekday
    ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    weekday_encoded = ohe.fit_transform(df[['weekday']])
    df = pd.concat([df, pd.DataFrame(weekday_encoded, columns=ohe.get_feature_names_out(['weekday']))], axis=1)
    df['week_number_in_month'] = df['Date'].apply(lambda x: (x.day-1) // 7 +1 )  # Week number in month
    df['week_sin'] = np.sin(2 * np.pi * df['week_number_in_month'] / 5)  # Assuming max 5 weeks
    df['week_cos'] = np.cos(2 * np.pi * df['week_number_in_month'] / 5)
    df['month_half'] = df['Date'].apply(lambda x: 1 if x.day <= 15 else 2)  # First or second half of the month
    df.dropna(inplace=True)
    return df
def split_feature(df):
    # X = df.loc[:,['d1', 'd2', 'd3', 'weekday_encoded', 'week_sin', 'week_cos']]  # Features
    y = df.loc[:,'day_classification']
    # 
    X = df.loc[:,['d1', 'week_sin', 'weekday_encoded']]  # Features
    
    
    # agggregate days in 2 groups: <2 and >=2

    y = y.apply(lambda x: 1 if 'high' in x else 0)
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y) # Stratify for class balance
    return X_train, X_test, y_train, y_test
df = create_lags(daily_summary)
X_train, X_test, y_train, y_test = split_feature(df)

# le = LabelEncoder()
# y_train_encoded = le.fit_transform(y_train)
# y_test_encoded = le.transform(y_test)

# Encode the features (d1, d2, d3)
le_features = LabelEncoder()
# for column in ['d1', 'd2', 'd3']:
for column in ['d1']:
    X_train[column] = le_features.fit_transform(X_train[column])
    X_test[column] = le_features.transform(X_test[column])
# Train a classifier (RandomForest is a good starting point)
rf = RandomForestClassifier(random_state=12, class_weight='balanced') # You can try other models (e.g., Gradient Boosting)
grid_search = GridSearchCV(rf, param_grid, cv=5, n_jobs=-1, scoring='balanced_accuracy', verbose=2)
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
model = grid_search.best_estimator_

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model (convert predictions back to original labels for reporting)
#y_pred_labels = le.inverse_transform(y_pred)
print("Classification Report:\n", classification_report(y_test, y_pred))

# Evaluate the model
#print(classification_report(y_test, y_pred))

# Feature Importance (to see which lags are most influential)
feature_importances = model.feature_importances_
print("Feature Importances:", feature_importances)


features = X_train.columns.tolist()


Fitting 5 folds for each of 81 candidates, totalling 405 fits
Best parameters: {'max_depth': 10, 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 200}
Classification Report:
               precision    recall  f1-score   support

           0       0.44      0.78      0.56         9
           1       0.71      0.36      0.48        14

    accuracy                           0.52        23
   macro avg       0.58      0.57      0.52        23
weighted avg       0.61      0.52      0.51        23

Feature Importances: [0.35220327 0.33326099 0.31453574]


In [191]:
X_train

,d1,week_sin,weekday_encoded
80,3,-0.951057,0.0
128,1,0.587785,0.0
166,2,0.587785,0.0
76,0,-0.951057,1.0
96,1,-0.951057,1.0
...,...,...,...
104,3,0.587785,2.0
146,1,0.587785,4.0
67,1,0.587785,3.0
136,1,-0.951057,4.0


In [147]:
X_train

,d1,week_sin,weekday_Monday,weekday_Thursday,weekday_Tuesday,weekday_Wednesday
87,2,5.877853e-01,0.0,0.0,0.0,0.0
60,2,-2.449294e-16,0.0,0.0,1.0,0.0
78,3,-9.510565e-01,1.0,0.0,0.0,0.0
69,1,5.877853e-01,1.0,0.0,0.0,0.0
94,3,-5.877853e-01,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...
95,3,-5.877853e-01,0.0,0.0,0.0,1.0
155,1,-9.510565e-01,0.0,0.0,0.0,0.0
143,1,9.510565e-01,0.0,0.0,0.0,1.0
100,3,9.510565e-01,0.0,0.0,0.0,1.0


In [156]:
import plotly.graph_objects as go

# Feature names and importances (based on your provided values and corrected order)
# features = ['d1', 'd2', 'd3', 'week_sin', 'week_cos', 'weekday_Friday', 'weekday_Monday', 
#             'weekday_Thursday', 'weekday_Tuesday', 'weekday_Wednesday']


importances = feature_importances
# Create a bar chart using Plotly
fig = go.Figure(data=[go.Bar(
    x=features,
    y=importances,
    marker_color='skyblue',  # Customize bar color
    text=importances,  # Display values on bars
    textposition='auto'  # Position text above bars
)])

# Customize the layout
fig.update_layout(
    title='Feature Importances in Random Forest Model',
    xaxis_title='Features',
    yaxis_title='Importance (Normalized)',
    xaxis={'tickangle': 45},  # Rotate x-axis labels for readability
    yaxis={'range': [0, 0.18]},  # Set y-axis range for better visibility
    bargap=0.2,  # Gap between bars
    plot_bgcolor='white',  # Background color
    showlegend=False  # No legend needed for a single bar chart
)

# Show the interactive plot
fig.show()


In [154]:
features

['d1',
 'd2',
 'd3',
 'week_sin',
 'week_cos',
 'weekday_Friday',
 'weekday_Monday',
 'weekday_Thursday',
 'weekday_Tuesday',
 'weekday_Wednesday']

In [157]:
fig = go.Figure(data=[go.Bar(
    y=features,
    x=importances,
    orientation='h',  # Horizontal bars
    marker_color='skyblue',
    text=importances,
    textposition='auto'
)])
fig.show()

In [181]:
from sklearn.inspection import permutation_importance
perm_importance = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42)
for i in range(len(features)):
    print(f"{features[i]}: {perm_importance.importances_mean[i]:.4f}")

d1: 0.1217
week_sin: 0.1652
weekday_encoded: 0.0783


In [182]:
import plotly.graph_objects as go

# Permutation importance values (from your report)

# Create a bar chart using Plotly
fig = go.Figure(data=[go.Bar(
    x=features,
    y=perm_importance.importances_mean,
    marker_color=[ 'green' if x >= 0 else 'red' for x in perm_importance.importances_mean],  # Green for positive, red for negative
    text=[f'{x:.4f}' for x in perm_importance.importances_mean],  # Display values on bars
    textposition='auto'  # Position text above bars
)])

# Customize the layout
fig.update_layout(
    title='Permutation Importances in Random Forest Model',
    xaxis_title='Features',
    yaxis_title='Permutation Importance (Change in Accuracy)',
    xaxis={'tickangle': 45},  # Rotate x-axis labels for readability
    yaxis={'range': [-0.07, 0.08]},  # Set y-axis range for better visibility
    bargap=0.2,  # Gap between bars
    plot_bgcolor='white',  # Background color
    showlegend=False  # No legend needed for a single bar chart
)

# Show the interactive plot
fig.show()

array([0.20416667, 0.09583333, 0.01666667, 0.0625    , 0.01666667,
       0.00416667])

In [6]:
dec = pd.read_pickle('0-2DTE_spy_options_01Dec23-31Dec23.pkl')
dec

,volume_options,volume_weighted_options,open_options,close_options,high_options,low_options,timestamp_options,number of trades_options,ticker,full_name,type,strike,expiry,equity_start_price,time_converted,open_equity,close_equity,high_equity,low_equity,timestamp_equity,number of trades_equity,_merge,equity_pct_change,options_pct_change,options_earliest_open
0,1,14.0200,14.02,14.02,14.02,14.02,1701441900000,1,SPY,O:SPY231201C00442000,call,442,2023-12-01,456.0443,2023-12-01 09:45:00,456.0443,456.1050,456.220,455.8100,1701441900000,18644,both,1.000000,1.0,14.02
1,3,14.9400,14.97,14.88,14.97,14.88,1701442800000,2,SPY,O:SPY231201C00442000,call,442,2023-12-01,456.0443,2023-12-01 10:00:00,456.1100,456.2800,457.130,455.9499,1701442800000,41350,both,1.000144,1.06776,14.02
2,1,14.0500,14.05,14.05,14.05,14.05,1701443700000,1,SPY,O:SPY231201C00442000,call,442,2023-12-01,456.0443,2023-12-01 10:15:00,456.2650,455.6900,456.350,455.6000,1701443700000,23136,both,1.000484,1.00214,14.02
3,1,14.1900,14.19,14.19,14.19,14.19,1701445500000,1,SPY,O:SPY231201C00442000,call,442,2023-12-01,456.0443,2023-12-01 10:45:00,456.1800,456.3300,456.490,456.1200,1701445500000,14206,both,1.000298,1.012126,14.02
4,3,15.2633,15.25,15.29,15.29,15.25,1701446400000,2,SPY,O:SPY231201C00442000,call,442,2023-12-01,456.0443,2023-12-01 11:00:00,456.3400,457.3400,457.410,455.1600,1701446400000,41644,both,1.000648,1.087732,14.02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57629,1,0.0100,0.01,0.01,0.01,0.01,1703867400000,1,SPY,O:SPY240102C00491000,call,491,2024-01-02,476.8750,2023-12-29 11:30:00,473.9400,474.1210,474.285,473.7700,1703867400000,24145,both,0.993845,1.0,0.01
57630,5,0.0100,0.01,0.01,0.01,0.01,1703873700000,1,SPY,O:SPY240102C00491000,call,491,2024-01-02,476.8750,2023-12-29 13:15:00,474.8800,474.5000,474.910,474.3600,1703873700000,17916,both,0.995817,1.0,0.01
57631,2,0.0100,0.01,0.01,0.01,0.01,1703879100000,2,SPY,O:SPY240102C00491000,call,491,2024-01-02,476.8750,2023-12-29 14:45:00,475.3450,475.5650,475.695,475.3200,1703879100000,17485,both,0.996792,1.0,0.01
57632,40,0.0100,0.01,0.01,0.01,0.01,1703881800000,2,SPY,O:SPY240102C00491000,call,491,2024-01-02,476.8750,2023-12-29 15:30:00,475.7700,475.3117,475.780,475.3100,1703881800000,31027,both,0.997683,1.0,0.01


In [16]:
sum(dec['timestamp_equity'] == 1701441900000)

123